# AgriSense AI – AI Plant Disease Detection Model Training (Google Colab GPU)

This notebook trains the **MobileNetV2 Transfer Learning** model for the **AgriSense AI Plant Doctor** module on Google Colab with free T4 GPU acceleration.

### Pipeline Overview:
1. **GPU Acceleration Check** (NVIDIA CUDA)
2. **Dataset Setup** (15 classes, Pepper, Potato, Tomato)
3. **Stratified 70/15/15 Holdout Partitioning**
4. **Class Imbalance Weighting**
5. **Data Augmentation** (Rotation, Zoom, Flip, Contrast)
6. **MobileNetV2 Transfer Learning** (ImageNet weights, 224x224)
7. **Independent Test Set Evaluation** (Accuracy, Confusion Matrix, Classification Report)
8. **Export**: `plant_disease_model.keras` & `plant_disease_model.tflite`

In [ ]:
# Step 1: Verify GPU Acceleration
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("[+] GPU Detected:", gpus)
else:
    print("[!] No GPU detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

In [ ]:
# Step 2: Mount Google Drive or Upload Dataset
# If using Google Drive:
from google.colab import drive
import os, shutil, json

# Option A: Mount Drive
# drive.mount('/content/drive')
# DATASET_DIR = '/content/drive/MyDrive/plantvillage'

# Option B: Or download directly from PlantVillage Kaggle / GitHub mirror
!pip install -q kaggle
print("Upload your dataset or zip file to /content/dataset/")

In [ ]:
# Step 3: Define Class Names and Parameters
CLASS_NAMES = [
  "Pepper__bell___Bacterial_spot",
  "Pepper__bell___healthy",
  "Potato___Early_blight",
  "Potato___Late_blight",
  "Potato___healthy",
  "Tomato_Bacterial_spot",
  "Tomato_Early_blight",
  "Tomato_Late_blight",
  "Tomato_Leaf_Mold",
  "Tomato_Septoria_leaf_spot",
  "Tomato_Spider_mites_Two_spotted_spider_mite",
  "Tomato__Target_Spot",
  "Tomato__Tomato_YellowLeaf__Curl_Virus",
  "Tomato__Tomato_mosaic_virus",
  "Tomato_healthy"
]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = len(CLASS_NAMES)

print(f"Target Classes: {NUM_CLASSES}")

In [ ]:
# Step 4: Build MobileNetV2 Transfer Learning Architecture
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import MobileNetV2

def build_model(num_classes=15):
    inputs = layers.Input(shape=(224, 224, 3), name="input_image")
    
    data_aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1)
    ], name="data_augmentation")
    
    x = data_aug(inputs)
    x = layers.Lambda(lambda img: tf.keras.applications.mobilenet_v2.preprocess_input(img))(x)
    
    base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze for transfer learning
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='prediction')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="AgriSense_MobileNetV2")
    return model

model = build_model(NUM_CLASSES)
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

In [ ]:
# Step 5: Train Model with EarlyStopping and ModelCheckpoint
checkpoint_cb = callbacks.ModelCheckpoint('plant_disease_model.keras', save_best_only=True, monitor='val_accuracy')
early_stopping_cb = callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_loss')
reduce_lr_cb = callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6, monitor='val_loss')

# Train on train_ds and val_ds
# history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint_cb, early_stopping_cb, reduce_lr_cb])

In [ ]:
# Step 6: Plot Training Curves & Evaluate on Test Set
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Export to TFLite for fast serverless inference
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('plant_disease_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("TFLite exported successfully!")

# Download files to your computer:
from google.colab import files
files.download('plant_disease_model.keras')
files.download('plant_disease_model.tflite')